# 05 - PatchCore baseline on the same 863 eval images (Week 3, part 2)

A compact re-implementation of PatchCore (Roth et al., CVPR 2022): WideResNet-50 features, a per-category memory bank of **good** patches from `train_good`, a 10% greedy coreset, and an image score = the largest distance from any patch to its nearest "normal" patch.

**Simplification (state it in the README):** no score re-weighting, so our numbers may land a little below the paper's ~99% image AUROC.

**Settings:** GPU T4 x2 · Internet On (to download ImageNet weights) · Inputs: **MVTec AD**, **mvtec-split**.
**Time:** roughly 20-40 minutes → run with **Save Version → Save & Run All**.

## Step 1 - Load the split

In [ ]:
import pandas as pd
from pathlib import Path

split_files = list(Path('/kaggle/input').rglob('splits.csv'))
assert split_files, "splits.csv not found: Add Input -> Your Work -> mvtec-split"
df = pd.read_csv(split_files[0])
roots = [p.parent for p in Path('/kaggle/input').rglob('bottle') if p.is_dir() and (p / 'train').exists()]
assert roots, "MVTec AD not found: Add Input -> the MVTec AD dataset"
ROOT = roots[0]

def rebase(p):
    if not isinstance(p, str):
        return p
    parts = p.split('/')
    i = parts.index('ground_truth') - 1 if 'ground_truth' in parts else len(parts) - 4
    return str(ROOT.joinpath(*parts[i:]))

df['path'] = df['path'].map(rebase)
df['mask_path'] = df['mask_path'].map(rebase)
assert Path(df.path.iloc[0]).exists(), "paths don't resolve"
print(len(df), 'rows |', df.role.value_counts().to_dict())

## Step 2 - PatchCore functions

In [ ]:
import torch, torch.nn.functional as F
import torchvision.transforms as T
from torchvision.models import wide_resnet50_2, Wide_ResNet50_2_Weights
from PIL import Image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TF = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class FeatureExtractor(torch.nn.Module):
    """WideResNet-50 layer2 + layer3, 3x3 local averaging, 1024-d patch features (as in the PatchCore paper/repo)."""
    def __init__(self, weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1, dim=1024):
        super().__init__()
        self.net = wide_resnet50_2(weights=weights).eval()
        self.dim = dim
    @torch.no_grad()
    def forward(self, x):
        n = self.net
        x = n.maxpool(n.relu(n.bn1(n.conv1(x))))
        f2 = n.layer2(n.layer1(x))            # B x 512 x 28 x 28
        f3 = n.layer3(f2)                     # B x 1024 x 14 x 14
        f2 = F.avg_pool2d(f2, 3, 1, 1)
        f3 = F.avg_pool2d(f3, 3, 1, 1)
        f3 = F.interpolate(f3, size=f2.shape[-2:], mode='bilinear', align_corners=False)
        f = torch.cat([f2, f3], 1)            # B x 1536 x 28 x 28
        B, D, H, W = f.shape
        f = f.permute(0, 2, 3, 1).reshape(-1, 1, D)
        f = F.adaptive_avg_pool1d(f, self.dim).reshape(B, H * W, self.dim)
        return f                              # B x 784 x 1024

def embed(extractor, paths, bs=16):
    out = []
    for i in range(0, len(paths), bs):
        x = torch.stack([TF(Image.open(p).convert('RGB')) for p in paths[i:i + bs]]).to(DEVICE)
        out.append(extractor(x).cpu())
    return torch.cat(out)                     # N x 784 x 1024

@torch.no_grad()
def greedy_coreset(feats, ratio=0.1, proj_dim=128, seed=0):
    """Greedy k-center selection on randomly projected features (PatchCore's coreset)."""
    g = torch.Generator().manual_seed(seed)
    n = len(feats); k = max(1, int(n * ratio))
    proj = torch.randn(feats.shape[1], proj_dim, generator=g).to(DEVICE)
    z = feats.to(DEVICE) @ proj
    idx = [int(torch.randint(n, (1,), generator=g))]
    dist = torch.cdist(z, z[idx]).squeeze(1)
    for _ in range(k - 1):
        j = int(torch.argmax(dist)); idx.append(j)
        dist = torch.minimum(dist, torch.cdist(z, z[j:j + 1]).squeeze(1))
    return feats[idx]

@torch.no_grad()
def image_scores(test_feats, bank, chunk=2048):
    """Image score = max over patches of the distance to the nearest memory-bank patch."""
    bank = bank.to(DEVICE); scores = []
    for f in test_feats:
        d = torch.cat([torch.cdist(f[i:i + chunk].to(DEVICE), bank).min(1).values
                       for i in range(0, len(f), chunk)])
        scores.append(d.max().item())
    return scores

## Step 3 - Fit and score each category
Memory bank from that category's `train_good` images only; scores only `eval` images. Neither model ever sees eval during training.

In [ ]:
import time
from tqdm.auto import tqdm
extractor = FeatureExtractor().to(DEVICE)
rows = []
t0 = time.time()
for cat in tqdm(sorted(df.category.unique()), desc='categories'):
    good = df[(df.category == cat) & (df.role == 'train_good')].path.tolist()
    ev = df[(df.category == cat) & (df.role == 'eval')]
    bank = greedy_coreset(embed(extractor, good).reshape(-1, extractor.dim), ratio=0.1)
    s = image_scores(embed(extractor, ev.path.tolist()), bank)
    rows += [dict(path=p, category=cat, kind=k, label=l, defect_type=d, score=v)
             for p, k, l, d, v in zip(ev.path, ev.kind, ev.label, ev.defect_type, s)]
    del bank; torch.cuda.empty_cache()
pc = pd.DataFrame(rows)
pc.to_csv('/kaggle/working/patchcore_eval_scores.csv', index=False)
print(f'{len(pc)} eval images scored in {(time.time() - t0) / 60:.1f} min')

## Step 4 - Quick look

In [ ]:
from sklearn.metrics import roc_auc_score
y = (pc.label == 'defect').astype(int)
print('PatchCore overall image AUROC:', round(roc_auc_score(y, pc.score), 3))
pc.groupby('category').apply(lambda g: round(roc_auc_score(g.label == 'defect', g.score), 3)).rename('AUROC')